In [1]:
import pandas as pd
df = pd.read_csv("sales_data.csv")
df.shape

(1030, 14)

In [2]:
before = len(df)
df = df.drop_duplicates()
print(before, "->", len(df))

1030 -> 1004


In [3]:
bad = df["quantity"] > 100
df.loc[bad, "quantity"] = (df.loc[bad, "total_sales"] / df.loc[bad, "unit_price"]).round().astype(int)
diff = (df["total_sales"] - df["quantity"] * df["unit_price"]).abs()
print("고친 줄 :", bad.sum(), " 수량 최대 :", df["quantity"].max(), " 총액 안 맞는 줄 :", (diff > 1).sum())

고친 줄 : 22  수량 최대 : 99  총액 안 맞는 줄 : 0


In [4]:
df["date_error"] = df["delivery_date"] < df["shipping_date"]
df["date_error"].value_counts()
df.to_csv("sales_clean.csv", index=False)
print(df["date_error"].sum(), "건,", df.shape)

147 건, (1004, 15)


In [5]:
df["shipping_date"] = pd.to_datetime(df["shipping_date"])
df["delivery_date"] = pd.to_datetime(df["delivery_date"])
days = (df.loc[~df["date_error"], "delivery_date"]
        - df.loc[~df["date_error"], "shipping_date"]).dt.days

print("평균:", days.mean())
print("중앙값:", days.median())
print(days[days > 300])

평균: 22.050469483568076
중앙값: 2.0
11     365.0
20     369.0
29     365.0
41     368.0
68     364.0
99     367.0
110    365.0
131    369.0
165    368.0
192    367.0
199    364.0
215    368.0
326    367.0
334    368.0
338    367.0
361    367.0
383    370.0
397    366.0
412    368.0
425    370.0
527    365.0
542    365.0
543    365.0
546    367.0
585    364.0
592    368.0
608    365.0
633    370.0
638    365.0
685    366.0
694    369.0
747    364.0
750    367.0
775    366.0
784    368.0
786    364.0
810    365.0
813    367.0
828    366.0
853    367.0
884    366.0
887    369.0
888    365.0
895    364.0
936    368.0
939    369.0
982    366.0
dtype: float64


In [6]:
return_rate = (
    df["status"].eq("반품")
    .groupby(df["satisfaction"])
    .mean() * 100
)

print(return_rate)

satisfaction
1.0     7.589286
2.0    11.956522
3.0     7.812500
4.0     8.720930
5.0     6.930693
Name: status, dtype: float64


In [7]:
result = df.groupby("category").agg(
    주문건수=("total_sales", "size"),
    총매출=("total_sales", "sum"),
    평균단가=("unit_price", "mean"),
    평균수량=("quantity", "mean")
)

print(result.sort_values("총매출", ascending=False))

          주문건수           총매출           평균단가       평균수량
category                                              
가구         213  2.843337e+09  257322.909061  51.042254
식품         210  2.513274e+09  255232.928143  47.314286
전자제품       207  2.471628e+09  249651.606135  47.961353
도서         204  2.453240e+09  238500.636765  49.495098
의류         170  2.181904e+09  249447.162235  51.611765
